In [10]:
import pandas as pd

import numpy as np
import re

In [9]:
# 📂 Đường dẫn tới file đã merge
csv_path = r"D:\Recommend_System\pre_data\vickypham_merged.csv"  

# Đọc file
df = pd.read_csv(csv_path)

# Hiển thị thông tin cơ bản
print("✅ Đã đọc dữ liệu thành công!")
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
display(df.head(3))


✅ Đã đọc dữ liệu thành công!
Số dòng: 450
Số cột: 18


,title,description,url,prep_time,cook_time,total_time,servings,ingredients,instructions,image_url,recipe_category,recipe_cuisine,keywords,calories,rating_value,rating_count,comment_count,review_count
0,Chicken Bamboo Noodle Soup (Bún Măng Gà),"Light, savory, and full of bamboo flavor and c...",https://vickypham.com/blog/chicken-bamboo-nood...,5m,50m,55m,7 servings,"[Ingredients] 1 lb bamboo , 3 lbs chicken (hal...","[Instructions] 1) [Prepare the bamboo, chicken...",https://i0.wp.com/vickypham.com/wp-content/upl...,entree,"Asian, Vietnamese","bun mang ga, vietnamese chicken bamboo soup, b...",NaN,5.0,1,2,3
1,Quick and Delicious Chicken Udon,This chicken udon is everything you want in a ...,https://vickypham.com/blog/chicken-udon/,5m,30m,35m,5 servings,"[Ingredients] 2 tablespoon sesame oil , 1 lbs ...",[Instructions] 1) [Pan fry the chicken and aro...,https://i0.wp.com/vickypham.com/wp-content/upl...,Entree,"Asian, Japanese","chicken udon, easy chicken udon recipe, chicke...",NaN,NaN,0,0,0
2,"Costco Salmon for Sashimi, Sushi & Poke","Trusting your senses and doing a bit of prep, ...",https://vickypham.com/blog/costco-salmon-sashi...,25m,0m,NaN,10 servings,[Ingredients] 3 lbs Costco farm raised Atlanti...,[Instructions] 1) [Shop early and inspect the ...,https://i0.wp.com/vickypham.com/wp-content/upl...,"entree, side dish","Asian, Japanese","costco salmon sushi, costco salmon sashimi, co...",NaN,NaN,0,0,0


In [11]:
import html

def unescape_multistep(x: str, max_steps: int = 3) -> str:
    """Giải mã HTML entities nhiều lần nếu bị double-escaped (&amp;amp;)."""
    prev = x
    for _ in range(max_steps):
        cur = html.unescape(prev)
        if cur == prev:
            break
        prev = cur
    return prev

AMP_PATTERN = re.compile(r"\s*&\s*")  # & có/không khoảng trắng hai bên

def normalize_text_field(s, mode: str = "decode"):
    """
    mode:
      - 'decode': &amp; -> &, giữ nguyên dấu &
      - 'and':    &amp; / & -> ' and '
    """
    s = s.fillna("").astype(str).map(unescape_multistep)
    # bỏ xuống dòng/tab -> 1 khoảng trắng
    s = s.str.replace(r"[\r\n\t]+", " ", regex=True)

    if mode == "and":
        s = s.map(lambda x: AMP_PATTERN.sub(" and ", x))

    # gộp khoảng trắng dư và strip
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

# ===== Áp dụng theo cột =====
if "title" in df.columns:
    # Thường muốn “and” để thống nhất tiêu đề/slug
    df["title"] = normalize_text_field(df["title"], mode="and")

if "description" in df.columns:
    # Mô tả: thường “and” cho đọc dễ + nhất quán
    df["description"] = normalize_text_field(df["description"], mode="and")

if "keywords" in df.columns:
    df["keywords"] = normalize_text_field(df["keywords"], mode="and")

# Ingredients/Instructions: thường chỉ cần decode, không ép thành "and"
if "ingredients" in df.columns:
    df["ingredients"] = normalize_text_field(df["ingredients"], mode="decode")

if "instructions" in df.columns:
    df["instructions"] = normalize_text_field(df["instructions"], mode="decode")

# TUYỆT ĐỐI không sửa URL / image_url (tránh hỏng link)
# if "url" in df.columns:      # bỏ qua
# if "image_url" in df.columns: # bỏ qua


In [12]:


# 🧽 Làm phẳng cột description (nếu có)
if "description" in df.columns:
    # giữ NaN là rỗng để không biến thành chuỗi "nan"
    desc = df["description"].fillna("")

    # giải mã HTML entities: &quot; -> "
    desc = desc.map(lambda x: html.unescape(str(x)))

    # thay xuống dòng/tab bằng 1 khoảng trắng
    desc = desc.str.replace(r"[\r\n\t]+", " ", regex=True)

    # gộp khoảng trắng lặp và strip 2 đầu
    desc = desc.str.replace(r"\s+", " ", regex=True).str.strip()

    df["description"] = desc


In [13]:
WHITELIST = {"M&M's"}
def protect_whitelist(x: str) -> str:
    return x if x in WHITELIST else AMP_PATTERN.sub(" and ", x)
df["title"] = df["title"].map(unescape_multistep).map(protect_whitelist)


In [14]:


# Các cột quan trọng phải có dữ liệu
required_cols = [
    "title",
    "description",
    "url",
    "ingredients",
    "instructions",
    "servings",
]

# Kiểm tra cột nào thực sự có trong DataFrame
required_cols = [c for c in required_cols if c in df.columns]

# Lọc: chỉ giữ dòng nào mà *tất cả* các cột quan trọng đều KHÔNG trống
df_clean = df.dropna(subset=required_cols)

# Ngoài ra, loại bỏ những dòng có title hoặc url trống hoàn toàn
df_clean = df_clean[(df_clean["title"].str.strip() != "") & (df_clean["url"].str.strip() != "")]

print(f"Đã xoá {len(df) - len(df_clean)} dòng thiếu dữ liệu quan trọng.")
print(f"Còn lại {len(df_clean)} dòng hợp lệ.")

# 🔍 Trước khi xoá trùng
print("Số dòng trước khi xoá trùng:", len(df_clean))

# 1️⃣ Xoá trùng theo cột 'title'
df_clean = df_clean.drop_duplicates(subset=["title"], keep="first")

# 2️⃣ Sau khi xoá
print("Số dòng sau khi xoá trùng:", len(df_clean))
print("🧹 Đã xoá", len(df) - len(df_clean), "dòng có title trùng lặp.")




df_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"🎉 Dữ liệu sạch đã được lưu tại: {csv_path}")



Đã xoá 61 dòng thiếu dữ liệu quan trọng.
Còn lại 389 dòng hợp lệ.
Số dòng trước khi xoá trùng: 389
Số dòng sau khi xoá trùng: 387
🧹 Đã xoá 63 dòng có title trùng lặp.
🎉 Dữ liệu sạch đã được lưu tại: D:\Recommend_System\pre_data\vickypham_merged.csv


In [15]:
df = pd.read_csv(csv_path)

In [16]:
df[["prep_time", "cook_time", "total_time"]].isnull().sum()

prep_time     10
cook_time     32
total_time     1
dtype: int64

In [17]:

# --- 1️⃣ Chuyển về phút ---
def to_minutes(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = re.sub(r"[^0-9hms ]", "", s)
    if "h" in s:
        h = re.search(r"(\d+)\s*h", s)
        m = re.search(r"(\d+)\s*m", s)
        hours = int(h.group(1)) if h else 0
        mins = int(m.group(1)) if m else 0
        return hours * 60 + mins
    m = re.search(r"(\d+)", s)
    return int(m.group(1)) if m else np.nan

for c in ["prep_time", "cook_time", "total_time"]:
    df[c] = df[c].apply(to_minutes)

# --- 2️⃣ Điền giá trị bị thiếu ---
mask_prep  = df["prep_time"].isna()  & df["total_time"].notna() & df["cook_time"].notna()
mask_cook  = df["cook_time"].isna()  & df["total_time"].notna() & df["prep_time"].notna()
mask_total = df["total_time"].isna() & df["prep_time"].notna()  & df["cook_time"].notna()

df.loc[mask_prep,  "prep_time"]  = df.loc[mask_prep,  "total_time"] - df.loc[mask_prep,  "cook_time"]
df.loc[mask_cook,  "cook_time"]  = df.loc[mask_cook,  "total_time"] - df.loc[mask_cook,  "prep_time"]
df.loc[mask_total, "total_time"] = df.loc[mask_total, "prep_time"]  + df.loc[mask_total, "cook_time"]

# --- 3️⃣ Nếu cook_time = 0 hoặc NaN → cook = total_time (nếu total_time > 0) ---
mask_cook_zero = ((df["cook_time"].isna()) | (df["cook_time"] == 0)) & (df["total_time"].notna()) & (df["total_time"] > 0)
df.loc[mask_cook_zero, "cook_time"] = df.loc[mask_cook_zero, "total_time"]

# --- 4️⃣ Nếu cook_time và total_time đều = 0 hoặc NaN ---
mask_both_zero = ((df["cook_time"].isna()) | (df["cook_time"] == 0)) & ((df["total_time"].isna()) | (df["total_time"] == 0))
# 👉 Chọn 1 trong 2 tuỳ mục đích:

# (A) Giữ lại nhưng ước lượng hợp lý (vd: cook = prep, total = prep * 2)
df.loc[mask_both_zero, "cook_time"]  = df.loc[mask_both_zero, "prep_time"]
df.loc[mask_both_zero, "total_time"] = df.loc[mask_both_zero, "prep_time"] * 2


# --- 5️⃣ Chuẩn hoá lại, không cho âm ---
df[["prep_time", "cook_time", "total_time"]] = df[["prep_time", "cook_time", "total_time"]].clip(lower=0)

# --- 6️⃣ Format về 'xxm' ---
def fmt_m(x):
    return f"{int(x)}m" if pd.notna(x) else ""

for c in ["prep_time", "cook_time", "total_time"]:
    df[c] = df[c].apply(fmt_m)

print("✅ Hoàn tất xử lý thời gian (không còn cook=0, total=0).")
display(df[["title", "prep_time", "cook_time", "total_time"]].head(3))




✅ Hoàn tất xử lý thời gian (không còn cook=0, total=0).


,title,prep_time,cook_time,total_time
0,Chicken Bamboo Noodle Soup (Bún Măng Gà),5m,50m,55m
1,Quick and Delicious Chicken Udon,5m,30m,35m
2,"Costco Salmon for Sashimi, Sushi and Poke",25m,25m,25m


In [18]:
df[["prep_time", "cook_time", "total_time"]].isnull().sum()

prep_time     0
cook_time     0
total_time    0
dtype: int64

In [19]:
def normalize_servings(x):
    """
    Nếu chỉ có số hoặc dải số (ví dụ '4' hoặc '4 - 6') -> thêm 'serving(s)'.
    Nếu có chữ (people, cupcakes, rolls, pieces, ...) -> giữ nguyên.
    """
    if pd.isna(x):
        return ""

    s = str(x).strip()
    # Nếu chứa chữ cái (people, pieces, muffins, ...), giữ nguyên
    if re.search(r"[a-zA-Z]", s):
        return s

     # Nếu là số thực (vd: 6.0, 3.00) -> chuyển về int string
    if re.fullmatch(r"\d+\.\d+", s):
        s_clean = str(int(float(s)))
        return s_clean + " serving(s)"
    
    # Nếu chỉ chứa số hoặc dấu '-' hoặc khoảng trắng -> thêm 'serving(s)'
    if re.fullmatch(r"[\d\s\-–]+", s):
        return s.strip() + " serving(s)"

    return s

# Áp dụng
df["servings"] = df["servings"].apply(normalize_servings)

print("✅ Hoàn tất chuẩn hoá cột servings.")
display(df[["title", "servings"]].head(3))


✅ Hoàn tất chuẩn hoá cột servings.


,title,servings
0,Chicken Bamboo Noodle Soup (Bún Măng Gà),7 servings
1,Quick and Delicious Chicken Udon,5 servings
2,"Costco Salmon for Sashimi, Sushi and Poke",10 servings


In [20]:
output_path = csv_path.replace(".csv", "_time-cleaned.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"🎉 File đã được lưu tại: {output_path}")


🎉 File đã được lưu tại: D:\Recommend_System\pre_data\vickypham_merged_time-cleaned.csv


### Clear Note

In [22]:
csv_path = r"D:\Recommend_System\pre_data\vickypham_merged_time-cleaned.csv"
df = pd.read_csv(csv_path)

print("✅ Số dòng:", len(df))
display(df[["title", "ingredients", "instructions"]].head(2))


✅ Số dòng: 387


,title,ingredients,instructions
0,Chicken Bamboo Noodle Soup (Bún Măng Gà),"[Ingredients] 1 lb bamboo , 3 lbs chicken (hal...","[Instructions] 1) [Prepare the bamboo, chicken..."
1,Quick and Delicious Chicken Udon,"[Ingredients] 2 tablespoon sesame oil , 1 lbs ...",[Instructions] 1) [Pan fry the chicken and aro...


In [23]:
def remove_notes(text):
    """
    Xoá toàn bộ cụm (Note ...), (note 1), (Note 3 re: ...), (Note:) ... — có thể xuất hiện nhiều lần trong 1 dòng.
    Không xoá nội dung trong [ ] hoặc các phần khác.
    """
    if not isinstance(text, str):
        return text

    # Xoá tất cả các cụm (Note ... ) — kể cả nhiều cụm liên tiếp
    cleaned = re.sub(r"\(\s*Note[^)]*\)", "", text, flags=re.IGNORECASE)

    # Có thể còn dấu ngoặc trống nếu note bị xoá giữa câu
    cleaned = re.sub(r"\(\s*\)", "", cleaned)

    # Làm sạch khoảng trắng dư & dấu phẩy thừa (ví dụ ", ,")
    cleaned = re.sub(r"\s{2,}", " ", cleaned)
    cleaned = re.sub(r",\s*,", ", ", cleaned)
    cleaned = re.sub(r"\s+,", ",", cleaned)
    cleaned = re.sub(r",\s+", ", ", cleaned)
    
    return cleaned.strip()


df["ingredients"] = df["ingredients"].apply(remove_notes)
df["instructions"] = df["instructions"].apply(remove_notes)

print("🎯 Đã xoá toàn bộ (Note ...) mà không ảnh hưởng đến [ ] hoặc nội dung khác.")
display(df[["title", "ingredients", "instructions"]].head(5))



🎯 Đã xoá toàn bộ (Note ...) mà không ảnh hưởng đến [ ] hoặc nội dung khác.


,title,ingredients,instructions
0,Chicken Bamboo Noodle Soup (Bún Măng Gà),"[Ingredients] 1 lb bamboo, 3 lbs chicken (half...","[Instructions] 1) [Prepare the bamboo, chicken..."
1,Quick and Delicious Chicken Udon,"[Ingredients] 2 tablespoon sesame oil, 1 lbs c...",[Instructions] 1) [Pan fry the chicken and aro...
2,"Costco Salmon for Sashimi, Sushi and Poke",[Ingredients] 3 lbs Costco farm raised Atlanti...,[Instructions] 1) [Shop early and inspect the ...
3,Crispy Vietnamese Oven-Roasted Pork Belly (Heo...,"[Ingredients] 3 lbs skin-on pork belly, 1 1/2 ...",[Instructions] 1) [Clean pork belly] Scrub por...
4,Crispy Salt and Pepper Tofu (Đậu Hũ Rang Muối),[Ingredients] 1 small shallot or 1/2 yellow on...,[Instructions] 1) [Prepare aromatics] Thinly s...


In [24]:
output_path = csv_path.replace(".csv", "_final.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"🎉 File đã được lưu tại:\n{output_path}")


🎉 File đã được lưu tại:
D:\Recommend_System\pre_data\vickypham_merged_time-cleaned_final.csv


## k

In [33]:
# ==============================
# 🍳 Normalize & Replace recipe_cuisine (Final - "All" fallback)
# Author: Jovana Golubovic
# ==============================

import pandas as pd
import re

# =======================================
# 1️⃣ Đọc file CSV gốc
# =======================================
src = r"D:\Recommend_System\data\recipes\all_recipes_weighted.csv"

df = pd.read_csv(src)
print(f"✅ Đã đọc file: {src}")
print(f"📦 Kích thước dữ liệu: {df.shape}")

if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong CSV!")

# =======================================
# 2️⃣ DANH SÁCH CẦN GÁN THÀNH "All"
# =======================================
drop_list = {"Dessert", "Baking", "Dog Food", "Universal"}

# =======================================
# 3️⃣ HÀM CHUẨN HOÁ CHUỖI
# =======================================
def normalize_name(name: str) -> str:
    if pd.isna(name):
        return ""
    return re.sub(r"\s+", " ", str(name).strip().lower())

# =======================================
# 4️⃣ TỪ ĐIỂN NHÓM CHUẨN
# =======================================
groups = {
    "Vietnamese": ["vietnamese", "modern vietnamese"],
    "Chinese": ["chinese", "sichuan", "szechuan"],
    "Japanese": ["japanese", "fusion japanese", "modern japanese"],
    "Korean": ["korean", "korean fusion"],
    "Thai": ["thai"],
    "Indian": ["indian"],
    "Italian": ["italian", "italian-esque", "italian american", "american italian"],
    "French": ["french", "french style", "french ish"],
    "British": ["british", "english", "uk", "scottish", "irish"],
    "American": [
        "american", "western", "western food", "southern", "south western", "tex mex", "tex-mex",
        "cajun", "creole", "new orleans", "thanksgiving", "christmas", "holiday", "festive",
        "bbq", "hawaiian", "louisiana"
    ],
    "Mexican": [
        "mexican", "mexican-esque", "latin", "latin american", "south american", "brazilian",
        "argentinian", "caribbean", "cuban", "trinidad"
    ],
    "Mediterranean": [
        "greek", "greek ish", "mediterranean", "mediterranean vibes", "lebanese", "turkish",
        "israeli", "persian", "middle eastern", "middle eastern ish", "middle eastern vibes",
        "arabic", "north african", "moroccan", "afghan"
    ],
    "European": [
        "european", "austrian", "german", "swiss", "hungarian", "polish", "spanish", "spanish style",
        "portuguese", "basque country", "nordic", "swedish", "russian", "russian ish", "bavarian"
    ],
    "Australian": ["australian", "aussie", "australia", "new zealand"],
    "Asian": [
        "asian", "asian-esque", "asian influence", "asian fusion", "asian fusioin", "south east asian",
        "southeast asian", "singaporean", "singapore", "malaysian", "taiwanese", "indonesian",
        "filipino", "lao", "bali", "modern asian", "fusion", "any flavour you want"
    ],
    "Jewish": ["jewish"],
    "African": ["african"]
}

# =======================================
# 5️⃣ PHÂN LOẠI & CHUẨN HOÁ
# =======================================
def normalize_cuisine(name: str) -> str:
    if pd.isna(name):
        return "All"

    name_stripped = str(name).strip()
    name_lower = name_stripped.lower()

    # Nếu là "All" hoặc thuộc drop_list → gán All luôn
    if name_lower == "all" or name_stripped in drop_list:
        return "All"

    name_l = normalize_name(name)
    assigned = []

    for group, keywords in groups.items():
        for kw in keywords:
            if kw in name_l:
                assigned.append(group)
                break

    # Multi-cuisine rules
    if "vietnamese" in name_l and "american" in name_l:
        return "Vietnamese, American"
    if "vietnamese" in name_l and "asian" in name_l:
        return "Vietnamese, Asian"
    if "chinese" in name_l and "american" in name_l:
        return "Chinese, American"
    if "italian" in name_l and "american" in name_l:
        return "Italian, American"
    if "korean" in name_l and "fusion" in name_l:
        return "Korean, Asian"
    if "fusion" in name_l and "japanese" in name_l:
        return "Japanese, Asian"

    # Nếu không xác định được → gán "All"
    if not assigned:
        return "All"

    return assigned[0] if len(assigned) == 1 else ", ".join(assigned)

# =======================================
# 6️⃣ THAY THẾ CỘT recipe_cuisine
# =======================================
df["cuisine_new"] = df["recipe_cuisine"].apply(normalize_cuisine)
df = df.drop(columns=["recipe_cuisine"])
df = df.rename(columns={"cuisine_new": "recipe_cuisine"})

# =======================================
# 7️⃣ LƯU FILE MỚI
# =======================================
out_path = r"D:\Recommend_System\data\recipes\all_recipes_weighted_normalized.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Đã thay thế hoàn toàn cột 'recipe_cuisine' (fallback = 'All') và lưu tại:\n{out_path}")

# Hiển thị mẫu để kiểm tra
print("\n🔎 Mẫu dữ liệu:")
print(df["recipe_cuisine"].head(25))


✅ Đã đọc file: D:\Recommend_System\data\recipes\all_recipes_weighted.csv
📦 Kích thước dữ liệu: (3556, 20)

✅ Đã thay thế hoàn toàn cột 'recipe_cuisine' (fallback = 'All') và lưu tại:
D:\Recommend_System\data\recipes\all_recipes_weighted_normalized.csv

🔎 Mẫu dữ liệu:
0     All
1     All
2     All
3     All
4     All
5     All
6     All
7     All
8     All
9     All
10    All
11    All
12    All
13    All
14    All
15    All
16    All
17    All
18    All
19    All
20    All
21    All
22    All
23    All
24    All
Name: recipe_cuisine, dtype: object
